# Logistic Regression — Fraud Detection Analysis
Canadian Fraud Dataset | RQ1, RQ2, RQ3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, f1_score, accuracy_score,
    precision_score, recall_score
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('canadian_fraud_cleaned.csv')
print(f'Dataset: {df.shape} | Fraud rate: {df["is_fraud"].mean():.4%}')
print(df['is_fraud'].value_counts())

## RQ2 — Behavioral Feature Engineering

In [ ]:
df['amt_log']          = np.log1p(df['amt'])
df['is_high_amt']      = (df['amt'] > df['amt'].quantile(0.95)).astype(int)
df['is_night']         = df['trans_hour'].apply(lambda h: 1 if h <= 5 or h >= 22 else 0)
df['is_weekend']       = df['trans_day_of_week'].apply(lambda d: 1 if d >= 5 else 0)
df['amt_per_month']    = df['amt'] / (df['Months_on_book'] + 1)
df['new_customer']     = (df['Months_on_book'] < 12).astype(int)
df['amt_income_ratio'] = df['amt'] / (df['Income_Category'] + 2)

behavioral_features = ['amt_log', 'is_high_amt', 'is_night', 'is_weekend',
                       'amt_per_month', 'new_customer', 'amt_income_ratio']

print('Behavioral feature means — Fraud vs Legit:')
print(df.groupby('is_fraud')[behavioral_features].mean().round(4).to_string())

In [ ]:
base_feature_cols = [c for c in df.columns if c not in ['is_fraud'] + behavioral_features]
full_feature_cols = [c for c in df.columns if c != 'is_fraud']

X_base, X_full, y = df[base_feature_cols], df[full_feature_cols], df['is_fraud']

X_base_train, X_base_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y)
X_full_train, X_full_test, _, _ = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y)

scaler_base = StandardScaler()
X_base_train_sc = scaler_base.fit_transform(X_base_train)
X_base_test_sc  = scaler_base.transform(X_base_test)

scaler_full = StandardScaler()
X_full_train_sc = scaler_full.fit_transform(X_full_train)
X_full_test_sc  = scaler_full.transform(X_full_test)

print(f'Train: {X_base_train.shape[0]} | Test: {X_base_test.shape[0]}')
print(f'Train fraud rate: {y_train.mean():.4%} | Test fraud rate: {y_test.mean():.4%}')

## RQ3 (Part A) — Baseline LR: No SMOTE

In [ ]:
lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_full_train_sc, y_train)

y_pred_baseline = lr_baseline.predict(X_full_test_sc)
y_prob_baseline = lr_baseline.predict_proba(X_full_test_sc)[:, 1]

print('Baseline LR — No SMOTE, t=0.5')
print(classification_report(y_test, y_pred_baseline, target_names=['Legit', 'Fraud'], zero_division=0))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_baseline).ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp} | ROC-AUC: {roc_auc_score(y_test, y_prob_baseline):.4f}')

## RQ3 (Part B) — LR with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_full_train_sm, y_train_sm = smote.fit_resample(X_full_train_sc, y_train)
print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}')

lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_full_train_sm, y_train_sm)

y_pred_smote = lr_smote.predict(X_full_test_sc)
y_prob_smote = lr_smote.predict_proba(X_full_test_sc)[:, 1]

print('\nLR + SMOTE, t=0.5')
print(classification_report(y_test, y_pred_smote, target_names=['Legit', 'Fraud'], zero_division=0))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_smote).ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp} | ROC-AUC: {roc_auc_score(y_test, y_prob_smote):.4f}')

## RQ2 — Base vs Full Feature Set + Coefficients

In [ ]:
X_base_train_sm, y_base_train_sm = smote.fit_resample(X_base_train_sc, y_train)
lr_base_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_base_smote.fit(X_base_train_sm, y_base_train_sm)
y_pred_base = lr_base_smote.predict(X_base_test_sc)
y_prob_base = lr_base_smote.predict_proba(X_base_test_sc)[:, 1]

print('Feature Engineering Impact (LR + SMOTE)')
print(f'{"Metric":<12} {"Base":>8} {"+Behavioral":>13} {"Delta":>10}')
print('-' * 45)
for metric, b_val, f_val in [
    ('ROC-AUC',   roc_auc_score(y_test, y_prob_base),                   roc_auc_score(y_test, y_prob_smote)),
    ('F1',        f1_score(y_test, y_pred_base, zero_division=0),        f1_score(y_test, y_pred_smote, zero_division=0)),
    ('Recall',    recall_score(y_test, y_pred_base, zero_division=0),    recall_score(y_test, y_pred_smote, zero_division=0)),
    ('Precision', precision_score(y_test, y_pred_base, zero_division=0), precision_score(y_test, y_pred_smote, zero_division=0)),
]:
    print(f'{metric:<12} {b_val:>8.4f} {f_val:>13.4f} {f_val - b_val:>+10.4f}')

coef_df = pd.DataFrame({
    'Feature': full_feature_cols,
    'Coefficient': lr_smote.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print('\nTop 15 LR Feature Coefficients (SMOTE, Full Features)')
print(coef_df.head(15).to_string(index=False))

## RQ3 (Part C) — Threshold Tuning: LR + SMOTE

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.01)
threshold_results = []
for thresh in thresholds:
    y_pred_t = (y_prob_smote >= thresh).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_t).ravel()
    threshold_results.append({
        'Threshold': thresh,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'F1':        f1_score(y_test, y_pred_t, zero_division=0),
        'FPR':       fp_t / (fp_t + tn_t)
    })

thresh_df = pd.DataFrame(threshold_results)
print('Threshold Analysis — LR + SMOTE:')
print(thresh_df.to_string(index=False, float_format='%.4f'))

best_row       = thresh_df.loc[thresh_df['F1'].idxmax()]
optimal_thresh = best_row['Threshold']
y_pred_optimal = (y_prob_smote >= optimal_thresh).astype(int)
cm_optimal     = confusion_matrix(y_test, y_pred_optimal)
tn, fp, fn, tp = cm_optimal.ravel()
print(f'\nOptimal t={optimal_thresh:.2f}: Prec={best_row["Precision"]:.4f}  Recall={best_row["Recall"]:.4f}  F1={best_row["F1"]:.4f}')
print(classification_report(y_test, y_pred_optimal, target_names=['Legit', 'Fraud'], zero_division=0))
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

## RQ3 (Part D) — Threshold Tuning: LR No SMOTE

In [ ]:
threshold_results_baseline = []
for thresh in thresholds:
    y_pred_t = (y_prob_baseline >= thresh).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_t).ravel()
    threshold_results_baseline.append({
        'Threshold': thresh,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'F1':        f1_score(y_test, y_pred_t, zero_division=0),
        'FPR':       fp_t / (fp_t + tn_t)
    })

thresh_df_baseline      = pd.DataFrame(threshold_results_baseline)
print('Threshold Analysis — LR No SMOTE:')
print(thresh_df_baseline.to_string(index=False, float_format='%.4f'))

best_row_baseline       = thresh_df_baseline.loc[thresh_df_baseline['F1'].idxmax()]
optimal_thresh_baseline = best_row_baseline['Threshold']
y_pred_optimal_baseline = (y_prob_baseline >= optimal_thresh_baseline).astype(int)
cm_optimal_baseline     = confusion_matrix(y_test, y_pred_optimal_baseline)
tn, fp, fn, tp = cm_optimal_baseline.ravel()
print(f'\nOptimal t={optimal_thresh_baseline:.2f}: Prec={best_row_baseline["Precision"]:.4f}  Recall={best_row_baseline["Recall"]:.4f}  F1={best_row_baseline["F1"]:.4f}')
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

## RQ1 — Final Summary: 4-Way Comparison

In [ ]:
print(f'{"Configuration":<45} {"Prec":>8} {"Recall":>8} {"F1":>8} {"AUC":>8} {"FPR":>8}')
print('-' * 85)
configs = {
    'No SMOTE (t=0.50)':                                        (y_pred_baseline,         y_prob_baseline),
    f'No SMOTE + Optimal (t={optimal_thresh_baseline:.2f})':    (y_pred_optimal_baseline, y_prob_baseline),
    'SMOTE (t=0.50)':                                           (y_pred_smote,            y_prob_smote),
    f'SMOTE + Optimal (t={optimal_thresh:.2f})':                (y_pred_optimal,          y_prob_smote),
}
best_f1, best_name = -1, ''
for name, (y_p, y_pr) in configs.items():
    tn, fp, fn, tp = confusion_matrix(y_test, y_p).ravel()
    f1_val = f1_score(y_test, y_p, zero_division=0)
    marker = ' <-- BEST F1' if f1_val > best_f1 else ''
    best_f1  = max(best_f1, f1_val)
    best_name = name if marker else best_name
    print(f'{name:<45} {precision_score(y_test, y_p, zero_division=0):>8.4f} '
          f'{recall_score(y_test, y_p, zero_division=0):>8.4f} {f1_val:>8.4f} '
          f'{roc_auc_score(y_test, y_pr):>8.4f} {fp/(fp+tn):>8.4f}{marker}')
print(f'\n>>> Best by F1: {best_name}')

## Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Logistic Regression — Fraud Detection Analysis', fontsize=16, fontweight='bold')

sns.heatmap(confusion_matrix(y_test, y_pred_baseline), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=axes[0, 0])
axes[0, 0].set_title('Confusion Matrix\n(Before SMOTE)', fontweight='bold')
axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_smote), annot=True, fmt='d',
            cmap='Oranges', xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=axes[0, 1])
axes[0, 1].set_title('Confusion Matrix\n(After SMOTE)', fontweight='bold')
axes[0, 1].set_xlabel('Predicted'); axes[0, 1].set_ylabel('Actual')

sns.heatmap(cm_optimal, annot=True, fmt='d',
            cmap='Greens', xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=axes[0, 2])
axes[0, 2].set_title(f'Confusion Matrix\n(SMOTE + Threshold={optimal_thresh:.2f})', fontweight='bold')
axes[0, 2].set_xlabel('Predicted'); axes[0, 2].set_ylabel('Actual')

fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_baseline)
fpr_s, tpr_s, _ = roc_curve(y_test, y_prob_smote)
axes[1, 0].plot(fpr_b, tpr_b, label=f'Before SMOTE (AUC={roc_auc_score(y_test, y_prob_baseline):.3f})', linewidth=2)
axes[1, 0].plot(fpr_s, tpr_s, label=f'After SMOTE (AUC={roc_auc_score(y_test, y_prob_smote):.3f})', linewidth=2)
axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1, 0].set_title('ROC Curve\n(Before vs After SMOTE)', fontweight='bold')
axes[1, 0].set_xlabel('False Positive Rate'); axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].legend()

prec_b, rec_b, _ = precision_recall_curve(y_test, y_prob_baseline)
prec_s, rec_s, _ = precision_recall_curve(y_test, y_prob_smote)
axes[1, 1].plot(rec_b, prec_b, label='Before SMOTE', linewidth=2)
axes[1, 1].plot(rec_s, prec_s, label='After SMOTE', linewidth=2)
axes[1, 1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1, 1].set_xlabel('Recall'); axes[1, 1].set_ylabel('Precision')
axes[1, 1].legend()

axes[1, 2].plot(thresh_df['Threshold'], thresh_df['Precision'], label='Precision', linewidth=2)
axes[1, 2].plot(thresh_df['Threshold'], thresh_df['Recall'],    label='Recall',    linewidth=2)
axes[1, 2].plot(thresh_df['Threshold'], thresh_df['F1'],        label='F1',        linewidth=2, linestyle='--')
axes[1, 2].axvline(x=optimal_thresh, color='red', linestyle=':', label=f'Optimal={optimal_thresh:.2f}')
axes[1, 2].set_title('Threshold Tuning\n(LR + SMOTE)', fontweight='bold')
axes[1, 2].set_xlabel('Threshold'); axes[1, 2].set_ylabel('Score')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('lr_analysis_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: lr_analysis_plots.png')

In [ ]:
fig2, axes2 = plt.subplots(1, 3, figsize=(20, 7))
fig2.suptitle('LR — RQ2 Behavioral Feature Evidence', fontsize=15, fontweight='bold')

fraud_means = df[df['is_fraud'] == 1][behavioral_features].mean()
legit_means = df[df['is_fraud'] == 0][behavioral_features].mean()
x = np.arange(len(behavioral_features))
w = 0.35
axes2[0].bar(x - w/2, legit_means.values * 100, w, label='Legit', color='#4ade80', alpha=0.85)
axes2[0].bar(x + w/2, fraud_means.values * 100, w, label='Fraud', color='#f87171', alpha=0.85)
axes2[0].set_xticks(x)
axes2[0].set_xticklabels(behavioral_features, rotation=35, ha='right', fontsize=9)
axes2[0].set_ylabel('Mean Value (%)')
axes2[0].set_title('Behavioral Feature Means\n(Fraud vs Legit)', fontweight='bold')
axes2[0].legend()
for i, (l, f) in enumerate(zip(legit_means.values, fraud_means.values)):
    axes2[0].text(i - w/2, l * 100 + 0.3, f'{l*100:.1f}%', ha='center', fontsize=7, color='#166534')
    axes2[0].text(i + w/2, f * 100 + 0.3, f'{f*100:.1f}%', ha='center', fontsize=7, color='#991b1b')

top_coef = coef_df.head(15).copy()
colors = ['#f87171' if c > 0 else '#60a5fa' for c in top_coef['Coefficient']]
axes2[1].barh(top_coef['Feature'], top_coef['Coefficient'], color=colors, alpha=0.85)
axes2[1].axvline(x=0, color='black', linewidth=0.8)
axes2[1].set_title('Top 15 LR Coefficients\n(SMOTE, Full Features)', fontweight='bold')
axes2[1].set_xlabel('Coefficient Value')
axes2[1].invert_yaxis()
for bar, val in zip(axes2[1].patches, top_coef['Coefficient']):
    axes2[1].text(val + (0.05 if val >= 0 else -0.05), bar.get_y() + bar.get_height()/2,
                  f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

metric_names = ['ROC-AUC', 'F1', 'Recall', 'Precision']
base_vals = [roc_auc_score(y_test, y_prob_base), f1_score(y_test, y_pred_base, zero_division=0),
             recall_score(y_test, y_pred_base, zero_division=0), precision_score(y_test, y_pred_base, zero_division=0)]
full_vals = [roc_auc_score(y_test, y_prob_smote), f1_score(y_test, y_pred_smote, zero_division=0),
             recall_score(y_test, y_pred_smote, zero_division=0), precision_score(y_test, y_pred_smote, zero_division=0)]
x3 = np.arange(len(metric_names))
axes2[2].bar(x3 - w/2, base_vals, w, label='Base Features', color='#94a3b8', alpha=0.85)
axes2[2].bar(x3 + w/2, full_vals, w, label='+ Behavioral',  color='#818cf8', alpha=0.85)
axes2[2].set_xticks(x3); axes2[2].set_xticklabels(metric_names)
axes2[2].set_ylim(0, 1.1); axes2[2].set_ylabel('Score')
axes2[2].set_title('Base vs Full Features\n(LR + SMOTE)', fontweight='bold')
axes2[2].legend()
for i, (b, f) in enumerate(zip(base_vals, full_vals)):
    axes2[2].text(i - w/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=8)
    axes2[2].text(i + w/2, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('lr_rq2_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: lr_rq2_plots.png')

In [ ]:
fig3, ax3 = plt.subplots(figsize=(9, 6))
fig3.suptitle('LR — SMOTE Impact at Default Threshold (t=0.50)', fontsize=13, fontweight='bold')

smote_metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
before_vals = [
    accuracy_score(y_test, y_pred_baseline),
    precision_score(y_test, y_pred_baseline, zero_division=0),
    recall_score(y_test, y_pred_baseline, zero_division=0),
    f1_score(y_test, y_pred_baseline, zero_division=0),
    roc_auc_score(y_test, y_prob_baseline),
]
after_vals = [
    accuracy_score(y_test, y_pred_smote),
    precision_score(y_test, y_pred_smote, zero_division=0),
    recall_score(y_test, y_pred_smote, zero_division=0),
    f1_score(y_test, y_pred_smote, zero_division=0),
    roc_auc_score(y_test, y_prob_smote),
]
x_si = np.arange(len(smote_metric_names))
w_si = 0.35
ax3.bar(x_si - w_si/2, before_vals, w_si, label='No SMOTE (t=0.50)', color='#94a3b8', alpha=0.85)
ax3.bar(x_si + w_si/2, after_vals,  w_si, label='SMOTE (t=0.50)',    color='#f97316', alpha=0.85)
ax3.set_xticks(x_si); ax3.set_xticklabels(smote_metric_names)
ax3.set_ylim(0, 1.15); ax3.set_ylabel('Score')
ax3.legend()
for i, (b, a) in enumerate(zip(before_vals, after_vals)):
    ax3.text(i - w_si/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=8)
    ax3.text(i + w_si/2, a + 0.02, f'{a:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('lr_smote_impact.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: lr_smote_impact.png')